In [ ]:
import os
import time
import pandas as pd
import dask.dataframe as dd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, lit, count, desc, to_timestamp, year, month, dayofweek, hour, avg, max, min, rank, lag, lead, row_number
from pyspark.sql.window import Window
from src.spark_session import get_spark_session
from src.data_cleaning import clean_data, drop_location_coordinates, fill_missing_values, extract_datetime_features


# 💾 File Paths Configuration


# Default project path configuration


In [ ]:
dataset_path = "data/Crimes_-_2001_to_Present.csv"
subset_path = "data/Crimes_2001_to_Present_subset.csv"
preprocessed_path = "data/Crimes_2001_to_Present_preprocessed.csv"
local_sample_path = "Crime_Data_from_2020_to_Present.csv"

# Fallback to local sample dataset if Kaggle file is not present
if not os.path.exists(dataset_path):
    print(f"[Pipeline Setup] Full dataset not found at '{dataset_path}'.")
    if os.path.exists(local_sample_path):
        print(f"[Pipeline Setup] Falling back to local sample file: '{local_sample_path}'")
        dataset_path = local_sample_path
        subset_path = "data/Crimes_2001_to_Present_subset_sample.csv"
        preprocessed_path = "data/Crimes_2001_to_Present_preprocessed_sample.csv"
    else:
        print("[Pipeline Setup] ERROR: Neither full dataset nor local sample was found.")

# Ensure output directory exists
os.makedirs("data", exist_ok=True)


# 📊 Section 1: Data Exploration (Pandas & Dask)


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 1: DATA EXPLORATION WITH PANDAS & DASK")
print("="*50)

if os.path.exists(local_sample_path):
    print(f"Loading local sample: {local_sample_path} using Pandas...")
    df_pd = pd.read_csv(local_sample_path)
    
    print("\nPandas Head (First 5 rows):")
    print(df_pd.head())
    
    print("\nPandas Describe (Basic Stats):")
    print(df_pd.describe())
    
    print("\nPandas Value Counts for 'Primary Type':")
    print(df_pd['Primary Type'].value_counts())
    
    print("\nConverting to Dask DataFrame (4 partitions)...")
    ddf = dd.from_pandas(df_pd, npartitions=4)
    print("Dask Partition Summary:")
    print(ddf)
    
    print("\nPlotting Domestic Crimes histogram...")
    sns.histplot(df_pd['Domestic'])
    plt.title("Domestic Crimes Distribution")
    plt.show()
else:
    print(f"Local sample file '{local_sample_path}' not found. Skipping Pandas & Dask exploration.")


# ⚙️ Section 2: PySpark Initialization & Loading


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 2: PYSPARK INITIALIZATION & DATA LOADING")
print("="*50)

spark = get_spark_session("Chicago Crime Analysis")
print("Spark Environment Configurations:")
for key, val in spark.sparkContext.getConf().getAll():
    print(f"  {key}: {val}")

print(f"\nLoading dataset in Spark from: {dataset_path}")
df = spark.read.csv(dataset_path, header=True, inferSchema=True)

print("\nPySpark DataFrame Schema:")
df.printSchema()

print("\nShowing first 5 rows (Spark):")
df.show(5)

print("\nShowing tail 5 rows (Spark):")
for r in df.tail(5):
    print(r)

print("\nDataset dimensions:")
print(f"  Total rows: {df.count()}")
print(f"  Total columns: {len(df.columns)}")

print(f"\nWriting raw subset out to: {subset_path}")
df.write.csv(subset_path, header=True, mode='overwrite')


# 🧹 Section 3: PySpark Data Cleaning Pipeline


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 3: STANDARDIZED PYSPARK DATA CLEANING PIPELINE")
print("="*50)

print("Dropping spatial coordinates features...")
df = drop_location_coordinates(df)
df.printSchema()

print("\nChecking missing values per column:")
missing_values = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
missing_values.show()

print("Filling missing values with calculated modes or defaults...")
df = fill_missing_values(df)

print("\nVerifying missing values after fill:")
missing_values_after_fill = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in ["Case Number", "Ward", "Community Area", "Location Description", "Location"]])
missing_values_after_fill.show()

print("Extracting datetime and temporal features...")
df = extract_datetime_features(df)

print(f"\nSaving preprocessed dataset to: {preprocessed_path}")
df.write.csv(preprocessed_path, header=True, mode='overwrite')


# 🔄 Section 4: PySpark RDD Operations


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 4: PYSPARK RDD OPERATIONS")
print("="*50)

print("Creating sample RDD from list...")
rdd_from_list = spark.sparkContext.parallelize([1, 2, 3, 4, 5])
rdd_from_df = df.select("Primary Type").rdd

print("Transforming RDD (map, filter, flatMap)...")
rdd_mapped = rdd_from_list.map(lambda x: x * 2)
rdd_filtered = rdd_mapped.filter(lambda x: x > 5)
rdd_flat = rdd_from_list.flatMap(lambda x: [x, x**2])

print("  Filtered RDD collected:", rdd_filtered.collect())
print("  RDD count:", rdd_from_list.count())
print("  RDD take(3):", rdd_from_list.take(3))
print("  RDD reduce sum:", rdd_from_list.reduce(lambda x, y: x + y))

print("\nKey-Value RDD operations (reduceByKey, groupByKey)...")
kv_rdd = spark.sparkContext.parallelize([("Theft", 1), ("Assault", 1), ("Theft", 1)])
reduced = kv_rdd.reduceByKey(lambda a, b: a + b)
grouped = kv_rdd.groupByKey().mapValues(list)
print("  Reduced Key-Value RDD:", reduced.collect())
print("  Grouped Key-Value RDD:", grouped.collect())

print("\nCalculating average yearly crimes by type using aggregateByKey...")
rdd = df.select("Primary Type", "Year").rdd.map(lambda row: ((row["Primary Type"], row["Year"]), 1))
rdd_agg = rdd.aggregateByKey(
    (0, 0),
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
)
rdd_avg = rdd_agg.mapValues(lambda x: x[0] / x[1])
print("First 5 average yearly crimes records:")
for record in rdd_avg.take(5):
    print(record)

print("\nSet operations on RDDs (Union, Intersection, Subtract):")
rdd1 = df.filter(F.col("Year") == 2020).select("Primary Type").rdd.distinct()
rdd2 = df.filter(F.col("Year") == 2021).select("Primary Type").rdd.distinct()
print("  Union:", rdd1.union(rdd2).distinct().collect())
print("  Intersection:", rdd1.intersection(rdd2).collect())
print("  Subtract (2020 not in 2021):", rdd1.subtract(rdd2).collect())


# 🔍 Section 5: PySpark SQL Queries & Spark SQL


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 5: PYSPARK SQL QUERIES & SPARK SQL VIEW")
print("="*50)

print("Joining Crime Types and Arrest Counts:")
df_type = df.groupBy("Primary Type").count().withColumnRenamed("count", "Total_Crimes")
df_arrest = df.filter(F.col("Arrest") == True).groupBy("Primary Type").count().withColumnRenamed("count", "Total_Arrests")
df_full = df_type.join(df_arrest, on="Primary Type", how="full")
df_full.show(10)

print("\nRanking Crime Types by Arrests using Window specs:")
df_rank = df.filter(F.col("Arrest") == True)\
            .groupBy("Primary Type")\
            .agg(F.count("*").alias("Total_Arrests"))
window_spec = Window.orderBy(F.desc("Total_Arrests"))
df_ranked = df_rank.withColumn("Rank", F.rank().over(window_spec))\
                   .withColumn("Row_Number", F.row_number().over(window_spec))\
                   .withColumn("Lag_Arrests", F.lag("Total_Arrests", 1).over(window_spec))\
                   .withColumn("Lead_Arrests", F.lead("Total_Arrests", 1).over(window_spec))
df_ranked.show(10)

print("\nTop 10 Crime Types by count:")
df.groupBy("Primary Type").count().orderBy(col("count").desc()).show(10)

print("\nSelecting projection columns:")
df.select("ID", "Primary Type", "Arrest", "Year").show(5)

print("\nFiltering arrests after 2020:")
df.filter((col("Arrest") == True) & (col("Year") >= 2020)).show(5)

print("\nCreating Temporary SQL View 'crime_data'...")
df.createOrReplaceTempView("crime_data")

print("Executing SQL query on Temp View:")
spark.sql("SELECT `Primary Type`, COUNT(*) AS Total FROM crime_data GROUP BY `Primary Type` ORDER BY Total DESC").show(10)

print("\nPerforming SQL Joins with custom datasets:")
df_clean_subset = df.dropna(subset=["Primary Type", "District", "Year"])
district_data = [
    (1, "Central"), (2, "Wentworth"), (3, "Grand Crossing"), (4, "South Chicago")
]
district_df = spark.createDataFrame(district_data, ["District", "District_Name"])

inner_join = df_clean_subset.join(district_df, on="District", how="inner")
print("  Inner Join Sample Results:")
inner_join.select("District", "District_Name", "Primary Type", "Year").show(5)

print("\nCalculating Yearly Crime aggregates:")
agg_df = df_clean_subset.groupBy("Year").agg(
    count("*").alias("Total_Crimes"),
    max("District").alias("Max_District"),
    min("District").alias("Min_District")
)
agg_df.orderBy("Year").show(5)

print("\nApplying Window partition ranks by Year:")
windowSpec = Window.partitionBy("Year").orderBy(col("District").desc())
ranked_df = df_clean_subset.select("Year", "District", "Primary Type") \
    .withColumn("Rank", rank().over(windowSpec)) \
    .withColumn("Row_Num", row_number().over(windowSpec)) \
    .withColumn("Prev_District", lag("District", 1).over(windowSpec)) \
    .withColumn("Next_District", lead("District", 1).over(windowSpec))
ranked_df.show(10)


# ⚡ Section 6: Caching, Persistence & Partitioning


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 6: CACHING, PERSISTENCE & PARTITIONING PERFORMANCE")
print("="*50)

# Initialize caching demo Spark session
spark_perf = get_spark_session("Caching_Persistence")

if os.path.exists(preprocessed_path):
    print(f"Loading preprocessed dataset from: {preprocessed_path}")
    df_perf = spark_perf.read.csv(preprocessed_path, header=True, inferSchema=True)
else:
    df_perf = df

df_filtered = df_perf.filter(F.col("Year") >= 2015)

print("Measuring performance without cache:")
start = time.time()
df_filtered.groupBy("Primary Type").count().show()
no_cache_time = time.time() - start

print("Caching DataFrame...")
df_filtered.cache()

print("Measuring performance with cache:")
start = time.time()
df_filtered.groupBy("Primary Type").count().show()
cache_time = time.time() - start
df_filtered.unpersist()

print(f"Performance: Without cache: {no_cache_time:.2f}s | With cache: {cache_time:.2f}s")

print("\nMeasuring count execution time without cache:")
start = time.time()
df_perf.filter(df_perf["Arrest"] == True).count()
end = time.time()
no_cache_count_time = end - start
print(f"  Time: {no_cache_count_time:.2f}s")

print("Caching DataFrame and materializing cache...")
df_perf.cache()
df_perf.count()

print("Measuring count execution time with cache:")
start = time.time()
df_perf.filter(df_perf["Arrest"] == True).count()
end = time.time()
cache_count_time = end - start
print(f"  Time: {cache_count_time:.2f}s")
df_perf.unpersist()

print("\nPartitioning checks...")
print(f"  Default partitions: {df_perf.rdd.getNumPartitions()}")

df_repart = df_perf.repartition(8)
print(f"  Partitions after repartitioning: {df_repart.rdd.getNumPartitions()}")

start = time.time()
df_repart.groupBy("Primary Type").agg(F.count("*").alias("Total")).collect()
end = time.time()
print(f"  Time taken with 8 partitions: {end - start:.2f}s")

df_coalesced = df_repart.coalesce(2)
start = time.time()
df_coalesced.groupBy("Primary Type").agg(F.count("*").alias("Total")).collect()
end = time.time()
print(f"  Time taken with 2 partitions: {end - start:.2f}s")

print("\nBroadcasting variables:")
important_types = ["THEFT", "BATTERY", "ASSAULT"]
broadcast_types = spark_perf.sparkContext.broadcast(important_types)
df_broadcast = df_perf.filter(F.col("Primary Type").isin(broadcast_types.value))
df_broadcast.select("Primary Type", "Description").show(5)

print("\nAccumulators implementation:")
acc_nulls = spark_perf.sparkContext.accumulator(0)
def count_nulls(row):
    global acc_nulls
    if row["Description"] is None:
        acc_nulls += 1
df_perf.foreach(count_nulls)
print("  Total null values in 'Description':", acc_nulls.value)

print("\nComparing Storage Levels (MEMORY_ONLY vs MEMORY_AND_DISK):")
df_filtered = df_perf.filter(F.col("Year") >= 2015)

df_filtered.persist()
start = time.time()
df_filtered.groupBy("Primary Type").count().collect()
mem_only_time = time.time() - start
df_filtered.unpersist()

df_filtered.persist(StorageLevel.MEMORY_AND_DISK)
start = time.time()
df_filtered.groupBy("Primary Type").count().collect()
mem_disk_time = time.time() - start
df_filtered.unpersist()

print(f"  MEMORY_ONLY: {mem_only_time:.2f}s | MEMORY_AND_DISK: {mem_disk_time:.2f}s")


# 🖼️ Section 7: Visualizations


In [ ]:
# ==========================================
print("\n" + "="*50)
print("SECTION 7: DATA VISUALIZATIONS")
print("="*50)

print("Preparing data and generating plots...")

# Convert SQL results to Pandas for visualization
top_crimes = df_perf.groupBy("Primary Type").count().orderBy(desc("count")).limit(10).toPandas()

plt.figure(figsize=(10, 5))
sns.barplot(x='Primary Type', y='count', data=top_crimes)
plt.xticks(rotation=45)
plt.title("Top 10 Crime Categories in Chicago")
plt.xlabel("Crime Type")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.show()

# Convert aggregated data to Pandas for visualization
agg_pd = agg_df.toPandas()

plt.figure(figsize=(10, 5))
sns.barplot(x='Year', y='Total_Crimes', data=agg_pd)
plt.title("Total Crimes per Year (Chicago)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(x='Year', y='Total_Crimes', data=agg_pd, marker="o")
plt.title("Crime Trends Over Time")
print("Line plot and bar charts rendered.")

plt.xlabel("Year")
plt.ylabel("Total Crimes")
plt.tight_layout()
plt.show()

crime_type_heat = (
    df_perf.groupBy("Year", "Primary Type")
      .count()
      .toPandas()
      .groupby(["Primary Type", "Year"])["count"].sum().unstack().fillna(0)
)

plt.figure(figsize=(14, 10))
sns.heatmap(crime_type_heat, cmap="viridis", linewidths=0.4)
plt.title("Crime Frequency by Type and Year", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Primary Type")
plt.tight_layout()
plt.show()

district_heat = (
    df_perf.groupBy("District", "Primary Type")
      .count()
      .toPandas()
      .groupby(["District", "Primary Type"])["count"].sum().unstack().fillna(0)
)

plt.figure(figsize=(14, 8))
sns.heatmap(district_heat, cmap="coolwarm", linewidths=0.4)
plt.title("Crime Type Distribution by District", fontsize=14)
plt.xlabel("Primary Type")
plt.ylabel("District")
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("PIPELINE COMPLETED SUCCESSFULLY!")
print("="*50)
